In [ ]:
using Pkg
repository = @__DIR__
while !isfile(joinpath(repository, "src", "Surge.jl"))
    parent = dirname(repository)
    parent == repository && error("Open this notebook from within the repository.")
    global repository = parent
end
Pkg.activate(joinpath(repository, "Starting_networks"))
include(joinpath(repository, "src", "Surge.jl"))
import .Surge
using JLD2
include(joinpath(repository, "src", "plotting_utils.jl"))

In [ ]:
experiment = 1
experiment_name = "FILM_long"
adaptation_method = :film              # :full, :output, :first_two, :last_two, :film, :lora
fold_count = 5
folds = collect(1:1)
repeats = collect(1:1)
training_event_count = 2
validation_event_count = 2
selection_seed = 42000
training_seed = 420000

domain = :uniform                     # :uniform, :interior, :boundary or :control
station_indices = [2,4,97,99]          # h observations only
# Other report layouts: one end [2,4,6,8]; eight [2,4,6,8,93,95,97,99]; spread [10,37,64,91].
dataset_file = joinpath(repository,"Data","drag_target_10.jld2")
pretrained_file = joinpath(
    repository,
    "Trained_models", "flux_correction_pretrained.bson",
)
output_directory = joinpath(repository, "results", "notebooks", "adaptation")
measurement_gap = 16
training_steps = 1000
forecast_hours = 72.0
observation_noise_std = 0.0             # metres, h only; no other regularisation in the report methods
training_noise_seed_offset = 710000
validation_noise_seed_offset = 720000
optimizer_seed_offset = 10000
optimizer_settings = (learning_rate=3e-6, betas=(0.9,0.999), epsilon=1e-8)
training_settings = (
    epochs=1000, batch_size=4, rollout_horizons=[16,32,64,128], window_stride=128,
    loss_mode=:balanced, endpoint_horizon=128, normalize_loss=true,
    curriculum_enabled=false, curriculum_boundaries=[0.25,0.375,0.5],
    detach_gradients=false, gradient_steps=16,
    validation_interval=5, patience=0, cache_refresh_interval=25,
)

save_loss_curves = true
animation_frames_per_second = 12
animation_maximum_frames = 180*animation_frames_per_second
nothing

In [ ]:
dataset = Surge.load_burgers_ensemble(basename(dataset_file);
    data_path=dirname(dataset_file), relative_times=true)
pretrained_model = Surge.load_model(pretrained_file)

In [ ]:
event_splits = Surge.make_event_splits(length(dataset.solution); fold_count=fold_count,
    folds=folds, repeats=repeats, training_event_count=training_event_count,
    validation_event_count=validation_event_count, selection_seed=selection_seed,
    training_seed=training_seed)

nothing

In [ ]:
ranks = nothing
if adaptation_method == :lora
    manual_lora_ranks = [2, 2, 2, 2, 2]

    ranks = Dict(
        fold => copy(manual_lora_ranks)
        for fold in 1:fold_count
    )
end
configuration = (
    experiment=experiment, name=experiment_name, domain=domain,
    output_directory=output_directory, dataset_file=dataset_file, pretrained_file=pretrained_file,
    measurement_gap=measurement_gap, training_steps=training_steps, stations=station_indices,
    observation_noise_std=observation_noise_std,
    training_noise_seed_offset=training_noise_seed_offset,
    validation_noise_seed_offset=validation_noise_seed_offset,
    optimizer_seed_offset=optimizer_seed_offset, optimizer=optimizer_settings,
    adaptation=(method=adaptation_method, ranks=ranks),
    training=training_settings,
)
nothing

In [ ]:
runs = Surge.run_experiment(pretrained_model, dataset, event_splits; configuration=configuration)
nothing

In [ ]:
evaluations = [Surge.evaluate_run(run, dataset, pretrained_model; forecast_hours=forecast_hours) for run in runs]
nothing

In [ ]:
for (run, evaluation) in zip(runs, evaluations)
    display(ExperimentPlots.plot_mse_heatmaps(run, evaluation))
end

In [ ]:
for (run, evaluation) in zip(runs, evaluations)
    display(ExperimentPlots.forecast_gif(run, evaluation;
        frames_per_second=animation_frames_per_second, maximum_frames=animation_maximum_frames))
end

In [ ]:
if save_loss_curves
    for run in runs
        display(ExperimentPlots.plot_losses(run))
    end
end

In [ ]:
JLD2.jldsave(joinpath(output_directory,experiment_name,"run_index.jld2");
    directories=[run.directory for run in runs], configuration=configuration,
    event_splits=event_splits, forecast_hours=forecast_hours)
nothing